# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR² Mass Spectrometry EV Human Protein Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All entities (record sets, fields, columns) are referenced by their `@id` fields as per best practices for Croissant-based data packages.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.date_published if hasattr(metadata, 'date_published') else 'N/A'}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and respective `@id`s for further analysis. All references use the internal `@id` for precise selection.

**Note**: If you are uncertain which record sets are present, you can examine `metadata.record_sets`, and then for each, list their fields and columns with their `@id`s.

In [ ]:
# List available record sets and their fields/columns using `@id` references
print("Available record sets (by @id):\n")
for rs in metadata.record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (field @id: {f.id})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (col @id: {col.id})")
    print()

## 3. Data Extraction
Load data from a specific record set (**using its `@id`**) into a DataFrame for further analysis.

Replace `<RECORD_SET_ID>` with your selected record set's `@id`. Multiple record sets can be loaded in a loop, as shown.

In [ ]:
# Find all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"Detected record sets: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set '{record_set_id}' with shape: {dataframes[record_set_id].shape}")

# Pick the first record set as an example for further analysis
primary_record_set_id = record_set_ids[0]
print(f"\nExample record set @id for analysis: {primary_record_set_id}")
print(f"Columns in '{primary_record_set_id}':\n{dataframes[primary_record_set_id].columns.tolist()}")
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, grouping, etc. using field `@id` or column names as appropriate.

To find a numeric field for demonstration (e.g. "coverage_percent", "PeptideCount_Total", "MW", etc.), inspect column names in the previous step. Replace `<NUMERIC_FIELD>` and `<GROUP_FIELD>` with field names or `@id` as shown in the overview.

In [ ]:
df = dataframes[primary_record_set_id]

# Try to use a common quantitative field if available
possible_numeric_fields = [
    'coverage_percent', 'PeptideCount_Total', 'MW', 'pI', 'abundance_normalized',
    'ProteinAbundanceSample1', 'ProteinAbundanceSample2', 'ProteinAbundanceSample3'
]
numeric_field = None
for f in possible_numeric_fields:
    if f in df.columns:
        numeric_field = f
        break
if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback
print(f"Selected numeric field: {numeric_field}")

# Filtering: Keep only proteins with value above threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalization (z-score)
filtered_df[numeric_field + "_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Optional: Group by a field (e.g. description or sample annotation)
possible_group_fields = ['description', 'modification', 'sample_name', 'accession']
group_field = next((f for f in possible_group_fields if f in df.columns), None)
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by '{group_field}':")
    display(grouped_df.head())
else:
    print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below are sample plots showing the distribution and relationships for the main quantitative field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If two numeric fields exist, show their relationship
if len(df.select_dtypes('number').columns) > 1:
    numeric_fields = df.select_dtypes('number').columns[:2]
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
    plt.title(f"{numeric_fields[0]} vs. {numeric_fields[1]}")
    plt.show()

## 6. Conclusion
- Loaded the Croissant-based FAIR² dataset and explored the schema using `@id` references.
- Inspected available record sets, extracted and analyzed key protein-related quantitative fields.
- Applied filtering and normalization, then visualized main distributions in the dataset.

For further analysis or ML, continue processing using the field names and `@id`s mapped in the data overview above.